# Praktikum Evaluasi Model Komprehensif pada Dataset Imbalanced

**Studi kasus:** Credit Card Fraud Detection  
**Cakupan:** poin A–P dari modul (poin Q/tugas praktikum tidak dikerjakan).

Tujuan utama adalah mendeteksi fraud dengan *recall* tinggi tanpa mengorbankan *precision* secara berlebihan. Selain ROC-AUC, notebook melaporkan PR-AUC karena lebih informatif untuk kelas positif yang sangat langka.

## A–B. Studi kasus dan pemuatan dataset

Dataset ULB berisi 284.807 transaksi dengan 492 fraud. Jika file belum tersedia, sel berikut mengunduhnya dari penyimpanan publik TensorFlow.

In [1]:
from pathlib import Path
from urllib.request import urlretrieve

DATA = Path('data/creditcard.csv')
DATA.parent.mkdir(exist_ok=True)
if not DATA.exists():
    url = 'https://storage.googleapis.com/download.tensorflow.org/data/creditcard.csv'
    print('Mengunduh dataset (sekitar 144 MB)...')
    urlretrieve(url, DATA)
print(f'Dataset: {DATA} ({DATA.stat().st_size / 1024**2:.1f} MB)')

Dataset: data/creditcard.csv (143.8 MB)


## C. Import library dan setup

In [2]:
from pathlib import Path
import json
import warnings

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, average_precision_score, confusion_matrix, f1_score,
    precision_recall_curve, precision_score, recall_score, roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import (
    GridSearchCV, StratifiedKFold, cross_validate, train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

warnings.filterwarnings("ignore")
np.random.seed(42)

ROOT = Path.cwd()
DATA = ROOT / "data" / "creditcard.csv"
OUT = ROOT / "outputs"
FIG = ROOT / "figures"
OUT.mkdir(exist_ok=True)
FIG.mkdir(exist_ok=True)


def metrics(y_true, y_pred, y_prob):
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "ROC_AUC": roc_auc_score(y_true, y_prob),
        "PR_AUC": average_precision_score(y_true, y_prob),
        "TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp),
    }


def save_cm(y_true, y_pred, title, filename):
    plt.figure(figsize=(6, 5))
    sns.heatmap(confusion_matrix(y_true, y_pred), annot=True, fmt="d", cmap="Blues",
                xticklabels=["Normal", "Fraud"], yticklabels=["Normal", "Fraud"])
    plt.title(title); plt.xlabel("Predicted"); plt.ylabel("Actual")
    plt.tight_layout(); plt.savefig(FIG / filename, dpi=150); plt.show()

## D. Load dataset dan eksplorasi awal

In [3]:
print("C-D. Load dataset dan eksplorasi")
df = pd.read_csv(DATA)
print("Shape:", df.shape)
print("Missing values:", int(df.isna().sum().sum()))
print("Distribusi kelas:\n", df["Class"].value_counts())
print(f"Proporsi fraud: {df['Class'].mean()*100:.4f}%")
print("Statistik Amount:\n", df["Amount"].describe())
print("Mean Amount fraud:", df.loc[df.Class == 1, "Amount"].mean())
print("Mean Amount normal:", df.loc[df.Class == 0, "Amount"].mean())

counts = df["Class"].value_counts().sort_index()
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(["Normal", "Fraud"], counts.values, color=["steelblue", "coral"])
axes[0].set_title("Distribusi Kelas"); axes[0].set_ylabel("Jumlah")
axes[1].pie(counts.values, labels=["Normal", "Fraud"], autopct="%1.3f%%",
            colors=["steelblue", "coral"])
axes[1].set_title("Proporsi Kelas")
plt.tight_layout(); plt.savefig(FIG / "01_distribusi_kelas.png", dpi=150); plt.show()

C-D. Load dataset dan eksplorasi


Shape: (284807, 31)
Missing values: 0
Distribusi kelas:
 Class
0    284315
1       492
Name: count, dtype: int64
Proporsi fraud: 0.1727%
Statistik Amount:
 count    284807.000000
mean         88.349619
std         250.120109
min           0.000000
25%           5.600000
50%          22.000000
75%          77.165000
max       25691.160000
Name: Amount, dtype: float64
Mean Amount fraud: 122.21132113821139
Mean Amount normal: 88.29102242231328


**Temuan awal:** akurasi berpotensi menipu karena fraud hanya sekitar 0,172%. Dataset tidak memiliki missing value.

## E. Preprocessing dan split data

In [4]:
print("\nE. Split data (scaling sengaja belum dilakukan agar tidak leakage)")
X = df.drop(columns="Class")
y = df["Class"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("Train:", X_train.shape, np.bincount(y_train))
print("Test :", X_test.shape, np.bincount(y_test))

all_results = []


E. Split data (scaling sengaja belum dilakukan agar tidak leakage)


Train: (227845, 30) [227451    394]
Test : (56962, 30) [56864    98]


Scaling belum dilakukan pada seluruh data. `StandardScaler` ditempatkan dalam pipeline agar hanya mempelajari statistik training set/fold dan mencegah leakage.

## F. Naive baseline dan akurasi yang menipu

In [5]:
print("\nF. Naive baseline")
y_pred_naive = np.zeros(len(y_test), dtype=int)
naive = metrics(y_test, y_pred_naive, y_pred_naive.astype(float))
all_results.append({"Strategi": "Naive Baseline", **naive})
print(naive)


F. Naive baseline
{'Accuracy': 0.9982795547909132, 'Precision': 0.0, 'Recall': 0.0, 'F1': 0.0, 'ROC_AUC': np.float64(0.5), 'PR_AUC': np.float64(0.0017204452090867595), 'TN': 56864, 'FP': 0, 'FN': 98, 'TP': 0}


Model yang selalu memilih kelas normal memperoleh akurasi hampir 100%, tetapi recall fraud = 0. Artinya seluruh transaksi fraud lolos.

## G–H. Evaluasi Logistic Regression dan confusion matrix

In [6]:
print("\nG-H. Logistic Regression baseline")
lr_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=2000, random_state=42)),
])
lr_pipe.fit(X_train, y_train)
y_pred_lr = lr_pipe.predict(X_test)
y_prob_lr = lr_pipe.predict_proba(X_test)[:, 1]
lr_result = metrics(y_test, y_pred_lr, y_prob_lr)
all_results.append({"Strategi": "LR Baseline", **lr_result})
print(lr_result)
save_cm(y_test, y_pred_lr, "Confusion Matrix – Logistic Regression", "02_cm_lr.png")


G-H. Logistic Regression baseline


{'Accuracy': 0.9991397773954567, 'Precision': 0.8266666666666667, 'Recall': 0.6326530612244898, 'F1': 0.7167630057803468, 'ROC_AUC': np.float64(0.9605494455801453), 'PR_AUC': np.float64(0.7413820992780461), 'TN': 56851, 'FP': 13, 'FN': 36, 'TP': 62}


## I. Handling imbalance dengan SMOTE

In [7]:
print("\nI. Logistic Regression + SMOTE (hanya training)")
smote_pipe = ImbPipeline([
    ("scaler", StandardScaler()),
    ("smote", SMOTE(random_state=42)),
    ("classifier", LogisticRegression(max_iter=2000, random_state=42)),
])
smote_pipe.fit(X_train, y_train)
y_pred_smote = smote_pipe.predict(X_test)
y_prob_smote = smote_pipe.predict_proba(X_test)[:, 1]
smote_result = metrics(y_test, y_pred_smote, y_prob_smote)
all_results.append({"Strategi": "LR + SMOTE", **smote_result})
print(smote_result)
save_cm(y_test, y_pred_smote, "Confusion Matrix – LR + SMOTE", "03_cm_smote.png")


I. Logistic Regression + SMOTE (hanya training)


{'Accuracy': 0.9741055440469084, 'Precision': 0.057803468208092484, 'Recall': 0.9183673469387755, 'F1': 0.10876132930513595, 'ROC_AUC': np.float64(0.9708434302252134), 'PR_AUC': np.float64(0.7244694356694709), 'TN': 55397, 'FP': 1467, 'FN': 8, 'TP': 90}


SMOTE hanya dijalankan pada training set melalui `imblearn.Pipeline`. Test set tetap merepresentasikan distribusi data nyata.

## J. Stratified 5-Fold Cross-Validation

In [8]:
print("\nJ. Stratified 5-fold cross-validation")
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = ["accuracy", "precision", "recall", "f1", "roc_auc", "average_precision"]
cv_rows = []
cv_models = {
    "Logistic Regression": lr_pipe,
}
for name, model in cv_models.items():
    scores = cross_validate(model, X_train, y_train, cv=skf, scoring=scoring, n_jobs=4)
    row = {"Model": name}
    for key in scoring:
        vals = scores[f"test_{key}"]
        row[f"{key}_mean"] = vals.mean()
        row[f"{key}_std"] = vals.std(ddof=1)
    cv_rows.append(row)
    print(name, row)

# Random Forest juga menggunakan sampel stratified agar notebook cepat direproduksi.
X_rf, _, y_rf, _ = train_test_split(
    X_train, y_train, train_size=40000, random_state=42, stratify=y_train
)
rf_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", RandomForestClassifier(n_estimators=30, random_state=42, n_jobs=1,
                                          class_weight="balanced_subsample")),
])
rf_scores = cross_validate(rf_pipe, X_rf, y_rf, cv=skf, scoring=scoring, n_jobs=4)
rf_row = {"Model": "Random Forest (sampel stratified 40k)"}
for key in scoring:
    vals = rf_scores[f"test_{key}"]
    rf_row[f"{key}_mean"] = vals.mean()
    rf_row[f"{key}_std"] = vals.std(ddof=1)
cv_rows.append(rf_row)
print(rf_row)

# RBF-SVM pada sampel stratified karena kompleksitas kernel kuadratik tidak realistis
# untuk keseluruhan 227 ribu baris training.
X_svm, _, y_svm, _ = train_test_split(
    X_train, y_train, train_size=10000, random_state=42, stratify=y_train
)
svm_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", SVC(kernel="rbf", probability=False, random_state=42)),
])
svm_scores = cross_validate(svm_pipe, X_svm, y_svm, cv=skf, scoring=scoring, n_jobs=4)
svm_row = {"Model": "SVM RBF (sampel stratified 10k)"}
for key in scoring:
    vals = svm_scores[f"test_{key}"]
    svm_row[f"{key}_mean"] = vals.mean()
    svm_row[f"{key}_std"] = vals.std(ddof=1)
cv_rows.append(svm_row)
print(svm_row)
pd.DataFrame(cv_rows).to_csv(OUT / "cv_models.csv", index=False)



J. Stratified 5-fold cross-validation


Logistic Regression {'Model': 'Logistic Regression', 'accuracy_mean': np.float64(0.9992012113498211), 'accuracy_std': np.float64(4.807852333866309e-05), 'precision_mean': np.float64(0.8760455835737655), 'precision_std': np.float64(0.043993243963736925), 'recall_mean': np.float64(0.6294060370009736), 'recall_std': np.float64(0.024944842060743995), 'f1_mean': np.float64(0.7315586528995093), 'f1_std': np.float64(0.012650636899151916), 'roc_auc_mean': np.float64(0.9783458160812424), 'roc_auc_std': np.float64(0.011022620467945998), 'average_precision_mean': np.float64(0.7614969310778783), 'average_precision_std': np.float64(0.045965510443186926)}


{'Model': 'Random Forest (sampel stratified 40k)', 'accuracy_mean': np.float64(0.9993000000000001), 'accuracy_std': np.float64(0.0003491060010942308), 'precision_mean': np.float64(0.8984848484848484), 'precision_std': np.float64(0.10841541709089889), 'recall_mean': np.float64(0.6626373626373626), 'recall_std': np.float64(0.1663211408495358), 'f1_mean': np.float64(0.7557246963562753), 'f1_std': np.float64(0.14958205746661876), 'roc_auc_mean': np.float64(0.9395425111342351), 'roc_auc_std': np.float64(0.07599032970002997), 'average_precision_mean': np.float64(0.7906971067681152), 'average_precision_std': np.float64(0.1847281944336079)}


/Users/ar/Project/MachineLearning/.venv/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/ar/Project/MachineLearning/.venv/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/ar/Project/MachineLearning/.venv/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", le

/Users/ar/Project/MachineLearning/.venv/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


{'Model': 'SVM RBF (sampel stratified 10k)', 'accuracy_mean': np.float64(0.9983000000000001), 'accuracy_std': np.float64(0.0002738612787526137), 'precision_mean': np.float64(0.0), 'precision_std': np.float64(0.0), 'recall_mean': np.float64(0.0), 'recall_std': np.float64(0.0), 'f1_mean': np.float64(0.0), 'f1_std': np.float64(0.0), 'roc_auc_mean': np.float64(0.9465422206455976), 'roc_auc_std': np.float64(0.10251718705162884), 'average_precision_mean': np.float64(0.7526504004470106), 'average_precision_std': np.float64(0.2066630127144225)}


RBF-SVM dievaluasi pada sampel stratified 10.000 baris karena kompleksitas kernel pada 227 ribu data training tidak praktis untuk notebook pembelajaran.

## K. Implementasi manual Stratified K-Fold

In [9]:
print("\nK. Manual Stratified 5-fold")
manual_rows = []
for fold, (tr, va) in enumerate(skf.split(X_train, y_train), start=1):
    model = clone(lr_pipe)
    model.fit(X_train.iloc[tr], y_train.iloc[tr])
    pred = model.predict(X_train.iloc[va])
    prob = model.predict_proba(X_train.iloc[va])[:, 1]
    row = {"Fold": fold, **metrics(y_train.iloc[va], pred, prob)}
    manual_rows.append(row)
    print(row)
manual_df = pd.DataFrame(manual_rows)
manual_df.to_csv(OUT / "manual_cv.csv", index=False)
print("Mean ± sample std:\n", manual_df[["Accuracy", "F1", "Recall", "ROC_AUC"]].agg(["mean", "std"]))



K. Manual Stratified 5-fold


{'Fold': 1, 'Accuracy': 0.9992319339902126, 'Precision': 0.9056603773584906, 'Recall': 0.6153846153846154, 'F1': 0.732824427480916, 'ROC_AUC': np.float64(0.9830158008149259), 'PR_AUC': np.float64(0.8196816824053996), 'TN': 45486, 'FP': 5, 'FN': 30, 'TP': 48}


{'Fold': 2, 'Accuracy': 0.9992319339902126, 'Precision': 0.9230769230769231, 'Recall': 0.6075949367088608, 'F1': 0.732824427480916, 'ROC_AUC': np.float64(0.9726371910922139), 'PR_AUC': np.float64(0.7566834216049007), 'TN': 45486, 'FP': 4, 'FN': 31, 'TP': 48}


{'Fold': 3, 'Accuracy': 0.999188044503939, 'Precision': 0.828125, 'Recall': 0.6708860759493671, 'F1': 0.7412587412587412, 'ROC_AUC': np.float64(0.9648563740535547), 'PR_AUC': np.float64(0.7032690892178198), 'TN': 45479, 'FP': 11, 'FN': 26, 'TP': 53}


{'Fold': 4, 'Accuracy': 0.9991222102745287, 'Precision': 0.8305084745762712, 'Recall': 0.620253164556962, 'F1': 0.7101449275362319, 'ROC_AUC': np.float64(0.9940980212649324), 'PR_AUC': np.float64(0.7925643877654285), 'TN': 45480, 'FP': 10, 'FN': 30, 'TP': 49}


{'Fold': 5, 'Accuracy': 0.9992319339902126, 'Precision': 0.8928571428571429, 'Recall': 0.6329113924050633, 'F1': 0.7407407407407407, 'ROC_AUC': np.float64(0.977121693180585), 'PR_AUC': np.float64(0.7352860743958433), 'TN': 45484, 'FP': 6, 'FN': 29, 'TP': 50}
Mean ± sample std:
       Accuracy        F1    Recall   ROC_AUC
mean  0.999201  0.731559  0.629406  0.978346
std   0.000048  0.012651  0.024945  0.011023


## L. GridSearchCV dengan pipeline

In [10]:
print("\nL. GridSearchCV")
# Validation terpisah juga dipakai pada bagian N agar threshold tidak dipilih di test set.
X_core, X_val, y_core, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42, stratify=y_train
)
param_grid = {
    "classifier__solver": ["liblinear"],
    "classifier__penalty": ["l1", "l2"],
    "classifier__C": [0.01, 0.1, 1.0, 10.0],
}
grid = GridSearchCV(lr_pipe, param_grid, cv=skf, scoring="f1", n_jobs=4,
                    return_train_score=False, verbose=1)
grid.fit(X_core, y_core)
print("Best params:", grid.best_params_)
print("Best CV F1:", grid.best_score_)
pd.DataFrame(grid.cv_results_).sort_values("rank_test_score").to_csv(
    OUT / "grid_search_results.csv", index=False
)

y_pred_best = grid.best_estimator_.predict(X_test)
y_prob_best = grid.best_estimator_.predict_proba(X_test)[:, 1]
best_result = metrics(y_test, y_pred_best, y_prob_best)
all_results.append({"Strategi": "Best GridSearch (threshold 0.5)", **best_result})
print("Test metrics:", best_result)



L. GridSearchCV


Fitting 5 folds for each of 8 candidates, totalling 40 fits


Best params: {'classifier__C': 1.0, 'classifier__penalty': 'l2', 'classifier__solver': 'liblinear'}
Best CV F1: 0.7642002969141968
Test metrics: {'Accuracy': 0.9991573329588147, 'Precision': 0.8289473684210527, 'Recall': 0.6428571428571429, 'F1': 0.7241379310344828, 'ROC_AUC': np.float64(0.9480033994464415), 'PR_AUC': np.float64(0.7440527141109519), 'TN': 56851, 'FP': 13, 'FN': 35, 'TP': 63}


## M. ROC Curve dan AUC

In [11]:
print("\nM. ROC Curve dan AUC")
fpr, tpr, _ = roc_curve(y_test, y_prob_best)
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, label=f"ROC-AUC = {best_result['ROC_AUC']:.4f}")
plt.plot([0, 1], [0, 1], "--", label="Random = 0.5")
plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
plt.title("ROC Curve – Model GridSearch"); plt.legend(); plt.grid(alpha=.3)
plt.tight_layout(); plt.savefig(FIG / "04_roc_curve.png", dpi=150); plt.show()



M. ROC Curve dan AUC


## N. Threshold tuning

In [12]:
print("\nN. Threshold tuning pada validation set (bukan test set)")
val_prob = grid.best_estimator_.predict_proba(X_val)[:, 1]
prec, rec, th = precision_recall_curve(y_val, val_prob)
f1s = 2 * prec * rec / (prec + rec + 1e-12)
valid_f1s = f1s[:-1]
idx = int(np.nanargmax(valid_f1s))
optimal_threshold = float(th[idx])
print("Optimal threshold:", optimal_threshold, "Validation F1:", valid_f1s[idx])
plt.figure(figsize=(9, 6))
plt.plot(th, prec[:-1], label="Precision")
plt.plot(th, rec[:-1], label="Recall")
plt.plot(th, valid_f1s, label="F1")
plt.axvline(optimal_threshold, color="black", ls="--", label=f"Optimal={optimal_threshold:.3f}")
plt.xlabel("Threshold"); plt.ylabel("Score"); plt.title("Threshold Tuning pada Validation Set")
plt.legend(); plt.grid(alpha=.3); plt.tight_layout()
plt.savefig(FIG / "05_threshold_tuning.png", dpi=150); plt.show()
y_pred_opt = (y_prob_best >= optimal_threshold).astype(int)
opt_result = metrics(y_test, y_pred_opt, y_prob_best)
all_results.append({"Strategi": f"Threshold Tuning ({optimal_threshold:.4f})", **opt_result})
print("Test metrics threshold optimal:", opt_result)



N. Threshold tuning pada validation set (bukan test set)
Optimal threshold: 0.05760576106400541 Validation F1: 0.7682119205293023


Test metrics threshold optimal: {'Accuracy': 0.9989993328885924, 'Precision': 0.6782608695652174, 'Recall': 0.7959183673469388, 'F1': 0.7323943661971831, 'ROC_AUC': np.float64(0.9480033994464415), 'PR_AUC': np.float64(0.7440527141109519), 'TN': 56827, 'FP': 37, 'FN': 20, 'TP': 78}


Threshold dipilih pada validation set, bukan test set. Dengan demikian test set tetap menjadi evaluasi akhir yang independen.

## O. Demonstrasi data leakage

In [13]:
print("\nO. Demonstrasi data leakage")
# Leakage sederhana mungkin hanya menghasilkan selisih kecil pada dataset besar ini.
wrong_scaler = StandardScaler()
X_wrong = wrong_scaler.fit_transform(X)
Xtr_w, Xte_w, ytr_w, yte_w = train_test_split(
    X_wrong, y, test_size=.2, random_state=42, stratify=y
)
wrong_model = LogisticRegression(max_iter=2000, random_state=42).fit(Xtr_w, ytr_w)
wrong_prob = wrong_model.predict_proba(Xte_w)[:, 1]
wrong_metrics = metrics(yte_w, wrong_model.predict(Xte_w), wrong_prob)
right_metrics = lr_result
leakage = pd.DataFrame([
    {"Cara": "SALAH: scaler fit seluruh data", **wrong_metrics},
    {"Cara": "BENAR: scaler dalam pipeline", **right_metrics},
])
print(leakage[["Cara", "Accuracy", "Precision", "Recall", "F1", "ROC_AUC"]])
leakage.to_csv(OUT / "leakage_comparison.csv", index=False)



O. Demonstrasi data leakage


                             Cara  Accuracy  Precision    Recall        F1  \
0  SALAH: scaler fit seluruh data  0.999157   0.828947  0.642857  0.724138   
1    BENAR: scaler dalam pipeline  0.999140   0.826667  0.632653  0.716763   

    ROC_AUC  
0  0.959947  
1  0.960549  


Selisih dapat sangat kecil pada dataset besar, tetapi cara pertama tetap salah: mean dan standar deviasi test set ikut memengaruhi preprocessing training.

## P. Perbandingan final semua strategi

In [14]:
print("\nP. Perbandingan final")
comparison = pd.DataFrame(all_results)
comparison.to_csv(OUT / "final_comparison.csv", index=False)
print(comparison.to_string(index=False))
plot_cols = ["Accuracy", "Precision", "Recall", "F1", "ROC_AUC", "PR_AUC"]
comparison.set_index("Strategi")[plot_cols].plot(kind="bar", figsize=(14, 7), colormap="Set2")
plt.title("Perbandingan Final Semua Strategi"); plt.ylabel("Score")
plt.xticks(rotation=25, ha="right"); plt.legend(loc="lower right", ncol=2)
plt.tight_layout(); plt.savefig(FIG / "06_final_comparison.png", dpi=150); plt.show()

summary = {
    "shape": list(df.shape),
    "fraud_count": int(y.sum()),
    "fraud_percent": float(y.mean() * 100),
    "best_params": grid.best_params_,
    "best_cv_f1": float(grid.best_score_),
    "optimal_threshold": optimal_threshold,
}
(OUT / "summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
print("\nSelesai. Semua output tersimpan di outputs/ dan figures/.")


P. Perbandingan final
                       Strategi  Accuracy  Precision   Recall       F1  ROC_AUC   PR_AUC    TN   FP  FN  TP
                 Naive Baseline  0.998280   0.000000 0.000000 0.000000 0.500000 0.001720 56864    0  98   0
                    LR Baseline  0.999140   0.826667 0.632653 0.716763 0.960549 0.741382 56851   13  36  62
                     LR + SMOTE  0.974106   0.057803 0.918367 0.108761 0.970843 0.724469 55397 1467   8  90
Best GridSearch (threshold 0.5)  0.999157   0.828947 0.642857 0.724138 0.948003 0.744053 56851   13  35  63
      Threshold Tuning (0.0576)  0.998999   0.678261 0.795918 0.732394 0.948003 0.744053 56827   37  20  78



Selesai. Semua output tersimpan di outputs/ dan figures/.


## Kesimpulan

Pemilihan model tidak boleh didasarkan pada accuracy saja. Baseline naif gagal total mengenali fraud. SMOTE menaikkan recall tetapi dapat menghasilkan lebih banyak false positive. GridSearch meningkatkan konfigurasi model, sedangkan threshold tuning memberi kontrol langsung terhadap trade-off precision–recall. Strategi akhir harus mengikuti biaya bisnis FN dan FP.